# Azure

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/azure/azure.ipynb)

This notebook covers the Azure services a data engineer actually uses: ADLS Gen2 and Blob Storage for the lake, Data Factory for orchestration and copy, Synapse SQL and Databricks for analytics and transformation, Event Hubs for streaming, and Entra ID with managed identities for security. The legacy Windows Azure cloud service material (web and worker roles, cspack, guest OS, swap deployments) that fills generic Azure interview lists is deliberately omitted — those services were retired or are irrelevant to data work. The *Things to remember* sections near the end compress the numbers and mistakes worth memorizing.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## The Azure mental model for data engineering

Azure's data platform maps cleanly onto the same five layers as any cloud lake. Learn the mapping once and every service fits a slot.

| Layer | Azure services |
|---|---|
| Ingest | Event Hubs (streams), Data Factory copy (batch), Azure Functions (small triggers) |
| Store | ADLS Gen2 (Blob Storage with hierarchical namespace), zones raw / curated / gold |
| Catalog & govern | Unity Catalog (Databricks), Microsoft Purview, Entra ID + RBAC + POSIX ACLs |
| Process | Databricks (Spark), Synapse Spark, Data Factory mapping data flows |
| Serve | Synapse dedicated SQL pool (warehouse), Synapse serverless SQL (ad-hoc over ADLS), Cosmos DB (low-latency lookups) |

Cross-cutting: orchestration (Data Factory triggers and pipelines), observability (Azure Monitor, diagnostic settings to ADLS), and cost control (DWU pause, auto-terminate clusters, storage tiers).

## Designing an Azure data lake

Use ADLS Gen2 (hierarchical namespace enabled) for storage, Azure Data Factory or Synapse Pipelines for orchestration, Databricks or Synapse Spark for processing, Purview for catalog and governance, and Synapse SQL or Power BI for serving. Organize storage into raw, curated, and serving layers.

```text
abfss://lake@acme.dfs.core.windows.net/raw/orders/ds=2026-10-04/
abfss://lake@acme.dfs.core.windows.net/curated/orders/ds=2026-10-04/
abfss://lake@acme.dfs.core.windows.net/marts/fact_orders/
```

Good answers include identity and security: managed identities instead of keys, RBAC plus POSIX ACLs on ADLS paths, private endpoints, Key Vault for secrets, and diagnostic logs to Log Analytics.

**Mental model.** A lake is organized by trust and refinement, not by source system. The usual zones are raw (landing, source-shaped), curated or silver (typed, deduplicated, conformed), and serving or gold (aggregated, modeled for a specific consumer). ADLS Gen2 with hierarchical namespace gives real directories and atomic directory renames, which object-only blob storage does not have.

**How it works.** Files land as Parquet or Delta, partitioned on the column that filters most often, usually a date. Managed identities read and write; POSIX ACLs on directories plus RBAC roles control who sees which zone. Lifecycle rules move cold partitions from Hot to Cool and then Archive, and Purview or Unity Catalog records schemas, owners, and lineage.

**Worked example.** Raw data sits at `abfss://lake@acmedata.dfs.core.windows.net/raw/orders/ds=2026-10-04/`, curated data is a Delta table at `curated/orders/`, and the serving layer is a Synapse dedicated table built by `CTAS`. A lifecycle rule moves `raw/orders/` to Cool after 30 days and deletes it after 365.

**Common mistake.** Building one flat container with no zones, or leaving every zone in the Hot tier forever. Both make the lake expensive and hard to govern. Another common slip is using the `blob` endpoint for a lake; directories there are only name prefixes, so renames and ACLs behave differently.

**Gotcha.** A lake is not a dumping ground. Without a catalog and a retention policy, raw, curated, and personal copies of the same table accumulate until nobody can tell which one is correct.

## Choosing the compute: Data Factory, Synapse, Databricks, or Functions

Use Data Factory for orchestration, copy activities, and connecting many sources. Use Synapse when you want integrated SQL, Spark, pipelines, and warehouse-style analytics in one workspace. Use Azure Databricks for collaborative Spark, Delta Lake, ML, and advanced tuning. Use Azure Functions for small event-driven automation.

Example choices:

+ nightly copy from SQL Server to ADLS: Data Factory
+ large Spark transformation over Delta tables: Databricks or Synapse Spark
+ SQL serving layer for Power BI: Synapse SQL
+ validate a file when it lands in blob storage: Azure Functions

**Mental model.** Pick the engine by the shape of the work, not by preference. Data Factory is an orchestrator and connector library: it moves data and schedules work, and it can run visual mapping data flows but is not built for heavy joins. Synapse covers SQL (serverless and dedicated pools) plus Spark. Databricks is a Spark and Delta platform tuned for large transformations and machine learning. Functions is small, event-driven glue.

**How it works.** Data Factory copy activities run on an integration runtime, so throughput is a configuration question (DIUs, parallelism, staging). Synapse dedicated pools store data in 60 distributions behind a clustered columnstore index and are sized in DWUs. Databricks separates driver and workers and bills DBUs per cluster-hour. Functions scales per event and has a time limit, so it suits webhooks and small files, not shuffles.

**Worked example.** Ingest 400 source tables with Data Factory (breadth of connectors), clean and join them in Databricks (heavy Spark work), publish marts to a Synapse dedicated pool for Power BI (concurrency and SQL semantics), and use a Function to validate a webhook payload.

**Common mistake.** Forcing one service to do everything: putting a large join in a mapping data flow, or running a long shuffle in Functions. Match the tool to the workload and let orchestration stay orchestration.

## Blob Storage: the object store under ADLS

A BLOB (Binary Large Object) is a file of any type and size in Azure Blob Storage. Blob Storage offers block blobs, append blobs, and page blobs. The URL format is `https://<account>.blob.core.windows.net/<container>/<blob>`, or `https://<account>.dfs.core.windows.net/<container>/<path>` for the ADLS Gen2 (DFS) endpoint.

**Mental model.** Blob Storage is Azure's object store: an account holds containers, a container holds blobs, and a blob is an immutable byte sequence addressed by a URL. It is the cheapest durable layer and the foundation of ADLS Gen2.

**How it works.** Blobs come in block, page, and append types. Durability comes from replication: locally redundant within a datacenter, zone redundant across zones, or geo redundant across regions. Access tiers (Hot, Cool, Cold, Archive) change storage cost against access cost, and lifecycle rules move data between them.

**Worked example.** A pipeline writes Parquet files to `raw/orders/ds=2026-10-04/`, and a lifecycle rule moves that prefix to Cool after 30 days and deletes it after 365.

**Common mistake.** Using blob storage as a database. Listing millions of small files is slow and expensive, and there is no transactional update. Compact files, partition paths, and keep an index in a real database or catalog.

## Block blobs vs page blobs

Block blobs are composed of blocks identified by block IDs; you upload blocks and commit them, and a small blob can be written in one `Put Blob`. They are optimized for sequential read/write and are the standard for data lakes, logs, and media. Page blobs are collections of 512-byte pages addressed by offset and are designed for random read/write, which is why they back Azure VM OS and data disks. Append blobs are optimized for append-only logs.

Limits to remember for interviews: a single block blob today supports up to about 190.7 TiB, and page blobs up to 8 TiB. If you need random writes to data, use a database or page blobs, not block blobs.

**Mental model.** Block blobs are built from committed blocks and are the default for files and analytics data. Page blobs are 512-byte-page random-access devices used to back virtual machine disks. Append blobs are optimized for append-only logs.

**How it works.** For a block blob you stage blocks and commit a block list, so a large upload can resume and commit atomically. For a page blob you write ranges; reads and writes are random access. Append blobs only allow appends, which makes them efficient for log tails but useless for rewriting.

**Worked example.** A log shipper writes to an append blob; a data pipeline writes gzipped or Parquet files as block blobs; a VM's OS disk is a page blob.

**Common mistake.** Choosing page blobs for analytical data or block blobs for a VM disk. The access pattern, not the size, decides the type.

## Queue Storage, Service Bus, and dead-letter queues

Azure Queue Storage is a simple REST-based get/put/peek queue for reliable persistent messaging within and between services. Service Bus Queues are part of a richer messaging infrastructure that supports queuing plus publish/subscribe, sessions, transactions, dead-lettering, duplicate detection, and integration patterns. Use Queue Storage for cheap high-volume decoupling and Service Bus for ordered, transactional, or topic-based messaging.

**Mental model.** Queue Storage is a simple, cheap, at-least-once queue with messages up to 64 KB and a maximum time to live of seven days. Service Bus is a messaging broker with ordering sessions, transactions, duplicate detection, dead-lettering, larger messages, and richer delivery semantics.

**How it works.** Queue Storage gives visibility timeouts and a poison-message count but no native ordering and no sessions. Service Bus adds FIFO with sessions, publish-subscribe through topics and subscriptions, scheduled messages, and dead-letter queues with a reason. Standard Service Bus messages are up to 256 KB; Premium goes to 100 MB.

**Worked example.** Use Queue Storage for a cheap image-thumbnail backlog; use Service Bus when an order must be processed in sequence per customer and a failed message must be parked with a reason.

**Common mistake.** Assuming Queue Storage preserves order or handles poisonous messages for you. After the dequeue count is exceeded you must move the message yourself, and ordering is not guaranteed.

A dead-letter queue holds messages the system could not process. Messages are moved there when:

+ A message expires and dead-lettering for expired messages is enabled.
+ The max delivery count is exceeded on a queue or subscription.
+ A filter evaluation exception occurs on a subscription and dead-lettering on filter exceptions is enabled.

**Mental model.** A dead-letter queue (DLQ) is where a broker parks messages it cannot deliver or that a consumer explicitly rejects. It is a safety net, not a trash can.

**How it works.** Service Bus moves messages to its DLQ on expiration, on exceeding the delivery count, on a filter evaluation error, or when the consumer calls dead-letter explicitly. Queue Storage has no DLQ, so you must implement one. The DLQ keeps the message body and the reason, so a message can be inspected and replayed after a fix.

**Worked example.** An order message fails five times because of a bad currency code. Service Bus dead-letters it with reason `MaxDeliveryCountExceeded`; an operator fixes the mapping, then a replay tool resubmits from the DLQ.

**Common mistake.** Letting the DLQ grow unwatched. Alert on DLQ depth, record the reason code, and build a replay path; otherwise failures are silent.

## Table Storage and Cosmos DB as sources

Azure Table Storage stores large amounts of structured, non-relational data and accepts authenticated calls from inside and outside Azure. A table is a collection of entities and does not enforce a schema; an account can have many tables. An entity is a set of properties like a row and can be up to 1 MB. A property is a name-value pair, up to 252 per entity, plus the system properties `PartitionKey`, `RowKey`, and `Timestamp`. Entities sharing a partition key are queried faster and support atomic batch operations; the row key is unique within a partition.

```python
from azure.data.tables import TableClient

table = TableClient.from_connection_string(conn_str, table_name="orders")
table.create_entity({"PartitionKey": "2026-10", "RowKey": "1001", "status": "created", "amount": 42.5})
```

Modern alternatives: Cosmos DB Table API for global scale and low latency, or Cosmos DB SQL API when you need richer queries and indexing.

**Mental model.** Table Storage is a schemaless key-value store where every entity is addressed by a partition key and a row key. Together they form the primary key and define locality: entities in the same partition are stored together.

**How it works.** There are no secondary indexes and no joins, and a query that does not constrain the partition key scans across partitions. Batch writes in the same partition are atomic (entity group transactions). The modern upgrade path is Azure Cosmos DB, whose Table API is compatible but adds global distribution and throughput controls.

**Worked example.** Store device telemetry as `PartitionKey = device_id`, `RowKey = inverted timestamp`, so the newest readings for a device are the first rows in one partition and read with a single range query.

**Common mistake.** Choosing a partition key with low cardinality, such as a status flag. All traffic lands in one partition, which creates a hot spot and throttles throughput. Also do not use Table Storage when you need secondary indexes or analytics.

## Writing a Synapse SQL warehouse pipeline

Synapse SQL has a serverless pool for querying files in place and dedicated pools for provisioned warehouse performance. Dedicated pools use distributions (hash, round-robin, replicated) and a clustered columnstore index; loads use `COPY INTO` from ADLS rather than row inserts.

```sql
-- Serverless: query Parquet directly from ADLS Gen2
SELECT TOP 10 *
FROM OPENROWSET(
  BULK 'https://acmedata.dfs.core.windows.net/lake/curated/orders/**',
  FORMAT = 'PARQUET'
) AS orders;

-- Dedicated pool: modeled table with hash distribution and clustered columnstore
CREATE TABLE dbo.fact_orders (
  order_id    BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  order_ts    DATETIME2 NOT NULL,
  amount      DECIMAL(12,2) NOT NULL
)
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX);

COPY INTO dbo.fact_orders
FROM 'https://acmedata.dfs.core.windows.net/lake/curated/orders/'
WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'));
```

**Mental model.** Synapse SQL has two engines. Serverless SQL queries files in ADLS in place and bills per terabyte scanned. Dedicated SQL pool is a provisioned MPP warehouse: every table is spread over 60 distributions, and you pay for DWUs whether you query or not. Choose dedicated for repeated, high-concurrency BI; choose serverless for exploration and occasional queries over the lake.

**How it works.** In a dedicated pool, a table's distribution and index decide how much data moves at query time. Hash distribution colocates rows with the same key so joins on that key avoid a shuffle. Replicated tables copy a small dimension to every distribution and remove the shuffle entirely. Round-robin spreads load evenly when there is no good join key, but every join then needs movement. The default index is a clustered columnstore, which compresses well and is fast for scans; heaps and clustered indexes are for staging and point lookups. Statistics tell the optimizer how many rows to expect, and a stale statistic produces a bad plan.

**Worked example.** A fact table joins a large customer dimension and a small date dimension on every query. Hash-distribute the fact on `customer_id`, replicate the date dimension, and keep the customer dimension hash-distributed on `customer_id` as well.

```sql
CREATE TABLE dbo.dim_date
WITH (DISTRIBUTION = REPLICATE, CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.dim_date;

CREATE TABLE dbo.fact_sales
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.fact_sales;

CREATE STATISTICS st_fact_sales_region ON dbo.fact_sales (region);
```

**Common mistake.** Distributing every table round-robin and then joining large tables. Each join adds a data movement step, and the plan fills with ShuffleMove operations that dominate runtime. A second common mistake is a clustered columnstore index on a table with only a few thousand rows: it lands in the delta store and performs worse than a heap.

## Building an Azure Data Factory pipeline

Data Factory pipelines are JSON activities driven by datasets and linked services. A copy activity reads a source dataset and writes a sink dataset, and triggers (schedule, tumbling window, or event) start the pipeline. Use parameterized datasets and a `ForEach` over a date range for backfills.

```json
{
  "name": "PL_Orders_Daily",
  "properties": {
    "parameters": {"windowStart": {"type": "String"}, "windowEnd": {"type": "String"}},
    "activities": [
      {
        "name": "CopyOrdersToCurated",
        "type": "Copy",
        "typeProperties": {
          "source": {
            "type": "ParquetSource",
            "storeSettings": {"type": "AzureBlobFSReadSettings", "recursive": true}
          },
          "sink": {
            "type": "ParquetSink",
            "storeSettings": {"type": "AzureBlobFSWriteSettings"}
          },
          "enableStaging": false
        },
        "inputs": [{"referenceName": "DS_RawOrders", "type": "DatasetReference",
          "parameters": {"folderPath": "@concat('raw/orders/ds=', pipeline().parameters.windowStart)"}}],
        "outputs": [{"referenceName": "DS_CuratedOrders", "type": "DatasetReference",
          "parameters": {"folderPath": "@concat('curated/orders/ds=', pipeline().parameters.windowStart)"}}]
      }
    ]
  }
}
```

Interview points: use managed identity for ADLS access, store secrets in Key Vault rather than linked-service credentials, prefer a tumbling-window trigger for exactly-once time slices, and use the data factory's mapping data flows only when a Spark-free visual transform is genuinely simpler than Databricks.

**Mental model.** Data Factory is a control plane: linked services describe connections, datasets describe data shapes, activities do work, and pipelines compose activities. An integration runtime is the compute that actually moves bytes. Azure IR is managed and serverless, a self-hosted IR runs on your own machines for on-premises and private networks, and Azure-SSIS IR runs SSIS packages.

**How it works.** A copy activity reads a source and writes a sink. Throughput is set by data integration units (DIUs), copy parallelism, and whether staging is enabled. Mapping data flows run on a Spark cluster and are billed by vCore-hour, so they suit visual transforms, not heavy joins. Triggers start pipelines: schedule, tumbling window (exactly-once time slices with state), and event-based (for example, blob created).

**Worked example.** Copy 2 TB of Parquet from an on-premises file share to ADLS in a four-hour window. Size the copy by DIUs and parallelism, and enable staging when the sink is a dedicated SQL pool.

```json
{
  "name": "CopyOrdersToCurated",
  "type": "Copy",
  "typeProperties": {
    "source": { "type": "ParquetSource" },
    "sink": { "type": "ParquetSink" },
    "dataIntegrationUnits": 32,
    "parallelCopies": 8,
    "enableStaging": true,
    "stagingSettings": {
      "linkedServiceName": { "referenceName": "LS_StagingLake", "type": "LinkedServiceReference" },
      "path": "staging/orders"
    }
  }
}
```

**Common mistake.** Raising DIUs while `parallelCopies` stays at one, or leaving a single self-hosted IR node for a very large copy. Both leave the bottleneck on the source reader. Also do not put a large join in a mapping data flow when Databricks is already in the stack; it duplicates cost and tooling.

## ADLS Gen2 paths and access control

ADLS Gen2 is Blob Storage with a hierarchical namespace, so paths behave like a file system and support POSIX-like ACLs at directory and file level, plus Azure RBAC at the account/container level. Use `abfss://` in Spark and `https://<account>.dfs.core.windows.net` in SQL and the CLI.

```text
abfss://lake@acmedata.dfs.core.windows.net/raw/orders/ds=2026-10-04/
abfss://lake@acmedata.dfs.core.windows.net/curated/orders/ds=2026-10-04/
```

```bash
az storage fs access set --account-name acmedata --file-system lake --path raw \
  --acl "user:11111111-2222-3333-4444-555555555555:r-x" --auth-mode login
az storage fs access show --account-name acmedata --file-system lake --path raw
```

**Mental model.** ADLS Gen2 is Blob Storage with hierarchical namespace enabled. Directories become real objects, so a directory rename is one atomic metadata operation instead of a copy of every blob. Access is controlled by two independent systems that are both evaluated: Azure RBAC at account, container, or path scope, and POSIX-style ACLs on directories and files.

**How it works.** A managed identity or user needs a data-plane role such as Storage Blob Data Contributor and, for restricted paths, an ACL entry that allows the operation. ACL evaluation walks from the container root down to the object; a missing execute permission on an intermediate directory blocks access even if the file has an allow entry. The mask on a directory limits what named users and groups can grant themselves.

**Worked example.** Grant a Synapse workspace identity write access to one zone only, without touching the rest of the lake.

```bash
az storage fs access set --account-name acmedata --file-system lake --path curated \
  --acl "user:<workspace-principal-id>:rwx" --auth-mode login

az role assignment create --assignee-object-id <workspace-principal-id> \
  --assignee-principal-type ServicePrincipal \
  --role "Storage Blob Data Contributor" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata
```

**Common mistake.** Granting the role at account scope and believing that secures the lake, or granting an ACL on a file without execute on its parent directories. Also remember that the `blob` endpoint does not expose directories, so ACLs only apply on the `dfs` endpoint.

**Gotcha.** Lifecycle management and ACLs are separate. A lifecycle rule can delete data that the team that owns it still expects to exist, so scope rules by prefix and test them on a small path first.

## Writing a Databricks notebook and PySpark job

In Databricks, notebooks are the interactive surface and `dbutils` provides workspace utilities. Jobs run the same code on a cluster; secrets come from secret scopes, and tables are usually Delta Lake managed by Unity Catalog. Prefer `MERGE` for idempotent upserts and `OPTIMIZE` for compaction.

```python
# Databricks notebook: read raw JSON from ADLS, type and dedupe, MERGE into a Delta table, then OPTIMIZE.
from pyspark.sql import functions as F

raw_path = "abfss://lake@acmedata.dfs.core.windows.net/raw/orders/"
orders = (
    spark.read.json(raw_path)
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .dropDuplicates(["order_id"])
)
orders.write.format("delta").mode("overwrite").saveAsTable("curated.orders_bronze")

spark.sql("""
merge into curated.orders as target
using curated.orders_bronze as source
on target.order_id = source.order_id
when matched then update set target.amount = source.amount, target.order_ts = source.order_ts
when not matched then insert *
""")
spark.sql("optimize curated.orders zorder by (customer_id)")
```

The same upsert logic runs locally in DuckDB, which supports `MERGE INTO` with the same semantics:

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE orders AS
    SELECT * FROM (VALUES (1, 10.00, TIMESTAMP '2026-10-03 09:00'), (2, 20.00, TIMESTAMP '2026-10-03 10:00'))
        t(order_id, amount, order_ts)
""")
# A raw batch with a duplicate redelivery, an update to order 2, and a new order 3.
con.execute("""
    CREATE TABLE raw_batch AS
    SELECT * FROM (VALUES
        (2, '25.00', '2026-10-04 08:00:00'), (2, '25.00', '2026-10-04 08:00:00'), (3, '30.00', '2026-10-04 09:30:00')
    ) t(order_id, amount, order_ts)
""")
con.execute("""
    CREATE TABLE bronze AS
    SELECT DISTINCT order_id, CAST(amount AS DECIMAL(12, 2)) AS amount, CAST(order_ts AS TIMESTAMP) AS order_ts
    FROM raw_batch
""")
con.execute("""
    MERGE INTO orders AS target
    USING bronze AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET amount = source.amount, order_ts = source.order_ts
    WHEN NOT MATCHED THEN INSERT (order_id, amount, order_ts) VALUES (source.order_id, source.amount, source.order_ts)
""")
result = con.execute("SELECT order_id, CAST(amount AS DOUBLE) FROM orders ORDER BY order_id").fetchall()
print(result)
assert result == [(1, 10.0), (2, 25.0), (3, 30.0)]

Interview points: Unity Catalog gives three-level names (`catalog.schema.table`) and centralized governance; Delta gives ACID and time travel; `zorder` speeds filtered reads on the column you filter most; Delta Live Tables or Lakeflow pipelines add declarative expectations and lineage.

**Mental model.** A Databricks notebook is the interactive surface; a job runs the same code on a schedule against a cluster. Clusters separate a driver from workers, and Delta Lake adds ACID transactions, schema enforcement, and time travel on top of Parquet in ADLS.

**How it works.** Jobs can autoscale workers between a minimum and maximum and can use job clusters that terminate when the run finishes. Unity Catalog gives three-level names (`catalog.schema.table`), centralized grants, and lineage. Delta maintenance keeps files healthy: `OPTIMIZE` compacts small files, and `ZORDER` clusters data on the columns you filter most. Photon accelerates SQL and DataFrame workloads at extra DBU cost.

**Worked example.** An ingest job reads JSON from `raw/`, types and deduplicates it, merges into a bronze Delta table on a business key, and then compacts the table.

**Common mistake.** Reading and writing many tiny files, or partitioning a small table by a high-cardinality column. Both create an explosion of files that slows every later job. Aim for files in the hundreds of megabytes and partition only on columns that are genuinely selectable and low enough in cardinality.

**Gotcha.** `mode("overwrite")` on a whole table is cheap to write and expensive to read later if it leaves thousands of small files behind. Follow bulk writes with compaction.

## Entra ID RBAC and managed identities

Microsoft Entra ID (formerly Azure AD) is the identity plane. Assign Azure roles at the right scope (management group, subscription, resource group, or resource) and use managed identities so services authenticate without stored secrets. For data access, the common roles are `Storage Blob Data Reader/Contributor`, `Synapse SQL Administrator`, and Databricks workspace/Unity Catalog grants.

```bash
# Grant a managed identity read access to the lake account at resource scope
az role assignment create --assignee-object-id <principal-id> --assignee-principal-type ServicePrincipal \
  --role "Storage Blob Data Contributor" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata

# Let a Synapse workspace use its managed identity to load data
az synapse workspace show --name syn-acme --resource-group rg-data --query identity.principalId
az storage fs access set --account-name acmedata --file-system lake --path curated \
  --acl "user:<principal-id>:rwx" --auth-mode login
```

**Mental model.** Microsoft Entra ID is the identity plane; Azure RBAC is the authorization plane. A principal (user, group, service principal, or managed identity) gets a role assignment at a scope, and permissions inherit down the scope tree from management group to resource. Managed identities give a service an Entra ID identity with no stored secret.

**How it works.** A system-assigned identity lives and dies with the resource; a user-assigned identity is a separate resource that several services can share. Workload identity federation lets a CI/CD pipeline exchange an external token for an Azure token without a client secret. Deny assignments override allows, and PIM makes privileged roles eligible rather than permanent.

**Worked example.** Give a Data Factory managed identity read access to one container, and confirm the effective assignment.

```bash
az datafactory show --name adf-acme --resource-group rg-data --query identity.principalId -o tsv

az role assignment create --assignee-object-id <principal-id> \
  --assignee-principal-type ServicePrincipal --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata/blobServices/default/containers/lake

az role assignment list --assignee <principal-id> --all -o table
```

**Common mistake.** Assigning roles to individual users instead of groups, and assigning at subscription scope when one container would do. Both make least privilege impossible to audit. Also remember that control-plane roles such as Contributor do not grant data-plane access; you still need a Storage Blob Data role and, for ADLS, the path ACLs.

**Gotcha.** A managed identity is not a security boundary on its own. Whoever can deploy to that resource can use its identity, so pair identities with scoped roles and with deployment controls.

## Streaming events with Azure Event Hubs

Event Hubs is the Azure streaming ingestion service: a partitioned event log with consumer groups, retention (1-90 days depending on tier), and capture to ADLS or Blob Storage. Producers send events with a partition key; consumers read with an offset using the EventProcessorClient or Spark Structured Streaming.

```python
from azure.eventhub import EventData, EventHubProducerClient

producer = EventHubProducerClient.from_connection_string(conn_str=connection_str, eventhub_name="orders")
with producer:
    batch = producer.create_batch(partition_key="customer-42")  # same key -> same partition -> ordered
    for order in orders:
        batch.add(EventData(json.dumps(order)))
    producer.send_batch(batch)
```

**Mental model.** Event Hubs is a partitioned, replayable event log, not a queue. Ordering is guaranteed within a partition only. Throughput units (Standard) or processing units (Premium) buy ingress and egress capacity, and the partition count sets the maximum consumer parallelism per consumer group.

**How it works.** A producer sends events with an optional partition key; the same key always lands in the same partition, which preserves per-key order. Consumers read from an offset in a consumer group, so several independent readers can each see the full stream. The recommended consumer uses the EventProcessorClient with a blob checkpoint store: it owns partitions through leases and records progress. Event Hubs Capture writes Avro micro-batches to ADLS or Blob Storage on a time and size window, so you get a durable log without writing a consumer.

## Choosing between Azure data services: trade-offs

+ ADLS Gen2 - durable lake storage with file semantics; not a database, no transactions or row-level updates.
+ Azure Data Factory - orchestrator and connector-heavy copy tool; not a compute engine for heavy transforms (use Databricks or Synapse Spark).
+ Azure Databricks - best for large Spark/Delta workloads, ML, and advanced tuning; heavier cost model and cluster management.
+ Synapse - unified SQL plus Spark plus pipelines; convenient when you already standardize on Synapse, but the Spark experience is generally less rich than Databricks.
+ Synapse SQL serverless - pay-per-query over files; poor for high-concurrency repeated BI.
+ Synapse SQL dedicated - high-concurrency warehouse; not for tiny point lookups or OLTP.
+ Azure Functions - small event-driven automation; avoid long-running or memory-heavy transforms.
+ Event Hubs - high-throughput streaming ingestion; not a task queue (use Service Bus or Storage Queues) and not a long-term store beyond retention.
+ Cosmos DB - globally distributed low-latency NoSQL; not for analytical scans or ad-hoc joins.
+ Microsoft Fabric - SaaS analytics end to end; strong when you want one governed platform, weaker when you need fine-grained infrastructure control.

Cost drivers: cluster DBUs and VM hours (Databricks), DWU/vCore hours and pause behavior (Synapse SQL), activity runs and data movement units (Data Factory), throughput units and retention (Event Hubs), storage transactions and tiering (ADLS), and provisioned RU/s (Cosmos DB). Security drivers: Entra ID RBAC, managed identities, POSIX ACLs, private endpoints, Key Vault, and Purview classification. Performance drivers: file size and Parquet/Delta layout, partitioning, distribution keys, shuffle, and cluster sizing.

## Designing a Synapse dedicated SQL pool for performance

**Mental model.** A dedicated SQL pool spreads every table across 60 distributions, and each query is planned as parallel work over those distributions. Performance is decided by what the plan has to move between them. If a join's keys are already colocated, the join runs locally on each distribution. If not, the engine inserts a data movement step, called a shuffle or broadcast, and that step often dominates runtime.

**How it works.** There are three distribution choices. `HASH` colocates rows by key, so joins and aggregations on that key stay local. `REPLICATE` copies the whole table to every distribution, which removes movement for a small dimension at the cost of storage and load time. `ROUND_ROBIN` spreads rows evenly with no key, which is fine for staging but forces movement on every join.

The index choice follows the workload. A clustered columnstore index compresses columnar data heavily and scans fast, and it is the default for fact tables. It pays off once a table is large enough to fill rowgroups, on the order of a million rows; below that, rows sit in the delta store and a heap or clustered index performs better. Heaps suit staging tables that are loaded and immediately read once. A clustered index suits point lookups and small tables.

Statistics are the optimizer's estimate of row counts, and a stale estimate produces a bad plan. Auto-create statistics is on, but you should create statistics explicitly on columns used in filters and joins, and update them after large loads. Result-set caching keeps the result of a repeated query on the control node; identical queries are served without touching distributions until the underlying data changes. Workload management groups queries into workload groups with resource percentages, and classifiers route a session into a group by user, role, or label, so an ETL job cannot starve interactive BI.

**Worked example.** Create a fact table, a replicated dimension, statistics on a filtered column, and a workload group for the ETL login.

```sql
CREATE TABLE dbo.fact_sales
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.fact_sales;

CREATE TABLE dbo.dim_date
WITH (DISTRIBUTION = REPLICATE, CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.dim_date;

CREATE STATISTICS st_fact_sales_region ON dbo.fact_sales (region);
UPDATE STATISTICS dbo.fact_sales;

CREATE WORKLOAD GROUP wg_etl WITH (MIN_PERCENTAGE_RESOURCE = 20, CAP_PERCENTAGE_RESOURCE = 40, REQUEST_MIN_RESOURCE_GRANT_PERCENT = 5);
CREATE WORKLOAD CLASSIFIER cls_etl WITH (WORKLOAD_GROUP = 'wg_etl', MEMBERNAME = 'etl_login');
```

**How to read the plan and the DMVs.** Start with `EXPLAIN` or the estimated execution plan in the portal, and look for `ShuffleMove`, `BroadcastMove`, and `TrimMove` operations. A large move on the biggest table usually means the distribution key does not match the dominant join.

The DMVs tell you where time went and whether the data is even. `sys.dm_pdw_exec_requests` lists requests and their status, `sys.dm_pdw_request_steps` breaks a request into steps, `sys.dm_pdw_sql_requests` shows each distribution's share, `sys.dm_pdw_dms_workers` reports per-worker data movement, and `sys.dm_pdw_nodes_db_partition_stats` gives row counts per distribution for skew checks. Workload group and classifier state is in `sys.workload_management_workload_groups` and `sys.workload_management_workload_classifiers`.

```sql
SELECT request_id, status, total_elapsed_time, submit_time
FROM sys.dm_pdw_exec_requests
WHERE status NOT IN ('Completed', 'Failed', 'Cancelled')
ORDER BY submit_time DESC;

SELECT distribution_id, SUM(row_count) AS rows_per_distribution
FROM sys.dm_pdw_nodes_db_partition_stats
WHERE object_id = OBJECT_ID('dbo.fact_sales')
GROUP BY distribution_id
ORDER BY rows_per_distribution DESC;
```

**Common mistake.** Hash-distributing on a column people filter on but do not join on, or round-robin distributing every table and then joining large tables. Both force a shuffle on every query. A second mistake is trusting a stale statistic: after a bulk load the optimizer may still assume a small table and choose a broadcast that overflows memory.

**Gotcha.** A replicated table must stay small. Replication multiplies the table by 60, so a dimension that grows past a few gigabytes slows down loads and wastes space across the whole pool.

In [ ]:
REPLICATE_MAX_GB = 2.0

fact = {"name": "fact_sales", "rows": 800_000_000, "size_gb": 240.0, "join_key": "customer_id"}
dimensions = [
    {"name": "dim_date", "rows": 7_300, "size_gb": 0.02, "join_key": "date_key"},
    {"name": "dim_customer", "rows": 4_000_000, "size_gb": 4.0, "join_key": "customer_id"},
    {"name": "dim_product", "rows": 250_000, "size_gb": 0.3, "join_key": "product_key"},
]


def choose_distribution(table, fact):
    if table["size_gb"] <= REPLICATE_MAX_GB:
        return "REPLICATE"
    if table["join_key"] == fact["join_key"]:
        return f"HASH({table['join_key']})"
    return "ROUND_ROBIN"


print("fact:", f"HASH({fact['join_key']})")
for dim in dimensions:
    print(f"{dim['name']:<14} {choose_distribution(dim, fact)}")

assert choose_distribution(dimensions[0], fact) == "REPLICATE"
assert choose_distribution(dimensions[1], fact) == "HASH(customer_id)"
assert choose_distribution(dimensions[2], fact) == "REPLICATE"

## Tuning Azure Data Factory copy performance

**Mental model.** A copy activity is a metered reader and writer. Throughput is the product of how many parallel streams you allow and how fast each stream runs, not the size of the source. Data integration units (DIUs) budget compute for a cloud copy; copy parallelism controls how many concurrent connections read and write; and staging lets a warehouse load use its fastest bulk path.

**How it works.** For a cloud-to-cloud copy, raise `dataIntegrationUnits` and `parallelCopies` together until the source or sink saturates. If the source is on-premises, the bottleneck is usually the self-hosted integration runtime: it runs on your network, and a large copy needs several IR nodes behind a single logical runtime so the work spreads. For file stores, partition the copy so each run handles a slice, either with a `ForEach` over folders or with partition options.

Staging matters when the sink is a Synapse dedicated pool. `enableStaging: true` writes the data to Blob or ADLS first and then loads it with PolyBase or `COPY INTO`, which is much faster than row-by-row insert. Mapping data flows are a different engine: they run on a Spark cluster and bill by vCore-hour, so they should do real transform work rather than a simple copy.

**Worked example.** A copy of Parquet from an on-premises share to ADLS with staging, explicit DIUs, and parallelism.

```json
{
  "name": "CopyOrdersToLake",
  "type": "Copy",
  "typeProperties": {
    "source": {"type": "ParquetSource", "storeSettings": {"type": "FileServerReadSettings", "recursive": true}},
    "sink": {"type": "ParquetSink", "storeSettings": {"type": "AzureBlobFSWriteSettings"}},
    "dataIntegrationUnits": 32,
    "parallelCopies": 8,
    "enableStaging": true,
    "stagingSettings": {
      "linkedServiceName": {"referenceName": "LS_StagingLake", "type": "LinkedServiceReference"},
      "path": "staging/orders"
    }
  }
}
```

```bash
az datafactory pipeline create-run --resource-group rg-data --factory-name adf-acme   --name PL_Orders_Daily --parameters '{"windowStart":"2026-10-04"}'

az datafactory activity-run query-by-pipeline-run --resource-group rg-data --factory-name adf-acme   --run-id <run-id> --last-updated-after 2026-10-04T00:00:00Z --last-updated-before 2026-10-05T00:00:00Z
```

A tumbling-window trigger is the right choice when each run owns one time slice and you need exactly-once behavior with backfill; a schedule trigger is simpler when runs can overlap harmlessly. Blob-created triggers are event-driven and good for landing-zone arrival, but they can fire several times for one file, so make the downstream write idempotent.

**Common mistake.** Raising DIUs while `parallelCopies` stays at one, or leaving a single self-hosted IR node for a multi-terabyte copy. Both leave the bottleneck at the reader. Another mistake is enabling staging without checking that the staging path is network-reachable and that the workspace identity can write to it.

**Gotcha.** More DIUs do not fix a slow source. Measure the source read rate and the sink write rate separately; whichever saturates first is the one to change.

In [ ]:
def dius_for_copy(volume_gb, window_minutes, mbps_per_diu=6.0, max_dius=256):
    mb = volume_gb * 1024
    seconds = window_minutes * 60
    needed = mb / (seconds * mbps_per_diu)
    return min(max_dius, int(needed) + 1)


small = dius_for_copy(2048, 240)
large = dius_for_copy(1024, 30)
print(f"2 TB in 4 h   -> {small} DIUs")
print(f"1 TB in 30 min -> {large} DIUs")

assert small == 25
assert large == 98
assert dius_for_copy(2048, 240, max_dius=16) == 16

## Sizing Event Hubs throughput and checkpointing

**Mental model.** Event Hubs scales on two axes: capacity units, which set the total bandwidth of the namespace, and partitions, which set the maximum parallelism of consumers within a consumer group. Capacity units are throughput units (TU) on Standard, processing units (PU) on Premium, and capacity units (CU) on Dedicated.

**How it works.** One Standard throughput unit carries up to 1 MB/s or 1,000 events per second of ingress, whichever comes first, and up to 2 MB/s or 4,096 events per second of egress. A namespace can hold up to 40 TUs, and auto-inflate raises the count when ingress grows. Premium is isolated per namespace and a single processing unit offers roughly 5 to 10 MB/s of ingress and 10 to 20 MB/s of egress; Dedicated gives whole capacity units. Partition count does not change cost, but it caps consumer parallelism: one epoch consumer owns a partition at a time, so extra readers sit idle.

Checkpointing is what makes replay safe. The recommended consumer is the EventProcessorClient with a blob-backed checkpoint store: it takes leases on partitions and writes the last processed offset, so a restart resumes near where it stopped. Checkpoint too often and you pay extra storage operations; too rarely and a crash replays a large batch. Backpressure appears as `ServerBusy` on the producer side when ingress exceeds purchased capacity, and as rising consumer lag when processors cannot keep up. Capture is the escape hatch: it writes Avro micro-batches to ADLS or Blob Storage on a time and size window, so the raw stream is durable even if a consumer is down.

**Worked example.** Size a namespace for 18 MB/s and 12,000 events per second of ingress with a 30 MB/s egress read, then add headroom.

```bash
az eventhubs namespace create --resource-group rg-data --name eh-acme --sku Standard --capacity 20 --enable-auto-inflate true --maximum-throughput-units 40

az eventhubs eventhub create --resource-group rg-data --namespace-name eh-acme \
  --name orders --partition-count 32 --message-retention 7

az eventhubs eventhub consumer-group create --resource-group rg-data \
  --namespace-name eh-acme --eventhub-name orders --name spark
```

Ingress needs 18 TUs from bandwidth and 12 TUs from event rate, so 18 is the floor. Egress of 30 MB/s needs 15 TUs, below ingress. With 1.5x headroom the target is 27 TUs, and Standard allows up to 40. Choose at least 27 partitions so one consumer per partition can keep up, and remember that partition count is fixed on Standard.

```python
def throughput_units(ingress_mbps, ingress_events_per_s, egress_mbps, headroom=1.0):
    ingress = max(ingress_mbps / 1.0, ingress_events_per_s / 1000.0)
    egress = egress_mbps / 2.0
    return max(ingress, egress) * headroom

print(round(throughput_units(18, 12000, 30, 1.5)))
assert round(throughput_units(18, 12000, 30, 1.5)) == 27
```

**Common mistake.** Sizing on average load instead of peak, so producers throttle with `ServerBusy` during the daily spike. Another is deploying more consumer instances than partitions and expecting more throughput; the extra instances stay idle.

**Gotcha.** A checkpoint records a position in the stream, not a transaction. If a processor crashes after processing and before checkpointing, those events are delivered again. Make the sink idempotent, or deduplicate on an event ID.

In [ ]:
def throughput_units(ingress_mbps, ingress_events_per_s, egress_mbps, headroom=1.0):
    ingress = max(ingress_mbps / 1.0, ingress_events_per_s / 1000.0)
    egress = egress_mbps / 2.0
    return max(ingress, egress) * headroom


peak = throughput_units(18, 12_000, 30, headroom=1.5)
partitions = 32
standard_cap = 40
print(f"ingress-limited TUs with 1.5x headroom = {peak:.0f}")
print(f"partitions = {partitions}, Standard TU cap = {standard_cap}")

assert round(peak) == 27
assert peak <= standard_cap
assert partitions >= round(peak)

## Laying out ADLS Gen2 files for Synapse and Spark

**Mental model.** Engines pay per file, not per byte. Every file adds a listing, a metadata read, and a task; thousands of tiny files turn a fast query into a scheduling problem. The layout that performs is a small number of large files, partitioned only on columns that queries actually filter.

**How it works.** Partition directories such as `ds=2026-10-04/` let a query prune whole prefixes, but only when the filter matches the partition column. Partitioning on a high-cardinality column creates millions of tiny directories, which is worse than no partitioning. For file size, aim for a few hundred megabytes per file; 128 MB is a common floor and 512 MB to 1 GB suits large scans. Compaction rewrites many small files into fewer large ones, and Delta Lake does this with `OPTIMIZE`.

ADLS Gen2 supports lifecycle rules that move data between Hot, Cool, Cold, and Archive. Cool has a 30-day minimum, Cold has a 90-day minimum, and Archive has a 180-day minimum; deleting or moving a blob before the minimum triggers an early deletion charge. Archive data cannot be read until it is rehydrated, which can take hours, so it belongs on raw or compliance data, never on a table a query might need.

**Worked example.** Query a partitioned Parquet prefix in serverless SQL, and archive the raw zone on a schedule.

```sql
SELECT ds, COUNT(*) AS rows_in_partition
FROM OPENROWSET(
  BULK 'https://acmedata.dfs.core.windows.net/lake/curated/orders/ds=*/',
  FORMAT = 'PARQUET'
) WITH (order_id BIGINT, customer_id BIGINT, amount DECIMAL(12,2), ds VARCHAR(10) '$.ds')
AS orders
GROUP BY ds;
```

```json
{
  "rules": [
    {
      "name": "raw-orders-tiering",
      "enabled": true,
      "type": "Lifecycle",
      "definition": {
        "filters": {"blobTypes": ["blockBlob"], "prefixMatch": ["lake/raw/orders/"]},
        "actions": {
          "baseBlob": {
            "tierToCool": {"daysAfterModificationGreaterThan": 30},
            "tierToArchive": {"daysAfterModificationGreaterThan": 180},
            "delete": {"daysAfterModificationGreaterThan": 365}
          }
        }
      }
    }
  ]
}
```

The curated zone stays Hot because queries read it daily, the raw zone moves to Cool and then Archive, and the deletion rule enforces retention. Serverless SQL bills per terabyte scanned, so a narrow projection and a matching partition filter are what keep the query cheap.

**Common mistake.** Landing every micro-batch as its own file, then querying the folder with serverless SQL. The engine spends more time opening files than reading them. A second mistake is partitioning a small table by a high-cardinality key such as an order ID.

**Gotcha.** Lifecycle policies are account-wide and irreversible for a blob that is already archived. Test a new rule against a small prefix first, and remember that a query against archived data fails until the blob is rehydrated.

In [ ]:
def plan_compaction(total_mb, file_count, target_mb=256):
    current_mb = total_mb / file_count
    target_files = max(1, -(-int(total_mb) // target_mb))
    return current_mb, target_files


total = 120_000 * 4
current, target = plan_compaction(total, 120_000)
print(f"current file size = {current:.0f} MB across 120,000 files")
print(f"after compaction   = {target} files of about {total / target:.0f} MB")
print(f"file-count reduction = {120_000 / target:.0f}x")

assert current == 4
assert target == 1875
assert round(120_000 / target) == 64

## Things to remember: key numbers and limits

Numbers that come up in design reviews. Prices are illustrative list prices at the time of writing; limits are stable.

| Topic | Number / fact | Why it matters |
|---|---|---|
| Block blob | up to ~190.7 TiB; composed of committed blocks | the default blob type for data files; page blobs are for VM disks |
| Queue Storage | messages up to 64 KB, at-least-once delivery, visibility timeout | a decoupling queue, not an analytics stream; use Event Hubs for replay |
| Event Hubs TU | 1 MB/s ingress and 2 MB/s egress per throughput unit; Standard caps at 40 TUs | size on peak, not average; partitions cap per-consumer-group parallelism |
| Event Hubs partitions | fixed at creation unless auto-inflate; ordering is per partition key | choose partition count for peak consumer parallelism |
| Synapse dedicated pool | every table spread across 60 distributions; replicate dimensions under ~2 GB | distribution choice decides whether big joins shuffle |
| Synapse concurrency | DW1000c allows 32 concurrent queries; scale up for more | concurrency, not raw speed, is often the sizing driver |
| ADF copy | about 6 MB/s per DIU, cap 256 DIUs; use staging for on-prem or large copies | compute DIUs from volume and window |
| ADLS tiers | Cool needs 30 days untouched, Cold 90, Archive 180; Archive rehydrate takes hours | tiering short-lived raw data wastes money on early-deletion fees |
| ADLS security | effective permission = union of RBAC and POSIX ACLs | an RBAC role can silently widen an ACL lockdown |
| File size target | roughly 128 MB to 1 GB Parquet files | engines pay per file in listing and planning, not per byte |

## Things to remember: semantics to memorize

* **Event Hubs is a replayable log, not a queue.** Queue Storage deletes on dequeue; Event Hubs retains for the retention period so consumers can rewind.
* **Managed identity over keys everywhere.** `--auth-mode login`, Key Vault for secrets, RBAC at the tightest scope; account keys in pipeline configs are the recurring incident.
* **RBAC and ACLs are additive on ADLS.** The effective permission is the union — tightening ACLs does not undo a broad RBAC grant.
* **Synapse serverless bills per TB processed**; a dedicated pool bills per hour while online — pause it or pay for idle weekends.
* **Dead-letter queues need a monitor and a replay path.** A growing DLQ is a silent outage; the messages are parked, not lost.
* **Distribution follows the dominant join** in a dedicated pool: hash on the join key, replicate small dimensions, round-robin only for staging.

## Real-world use cases on Azure

### Use case 1: Nightly retail ELT into a Synapse warehouse

Business context: a retailer consolidates point-of-sale, e-commerce, and inventory into one reporting layer for merchandising. Data volume: ~2 TB/night, ~40 source systems.

Architecture: Data Factory copies sources to `raw/` in ADLS Gen2, Databricks cleans and models to Delta in `curated/`, and Synapse SQL dedicated pools serve Power BI through `COPY INTO`.

```sql
COPY INTO dbo.fact_sales
FROM 'https://acmedata.dfs.core.windows.net/lake/curated/fact_sales/'
WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'));
```

Why: ADF handles breadth of connectors cheaply, Databricks absorbs the heavy transformation, and Synapse SQL gives sub-second dashboard queries with hash-distributed facts and replicated dimensions.

### Use case 2: IoT telemetry streaming with Event Hubs and Spark

Business context: a manufacturer monitors equipment health and predicts failures. Data volume: ~5 million events/minute, ~30 TB/month.

Architecture: devices -> IoT Hub/Event Hubs (8 or more partitions) -> Databricks Structured Streaming with a Delta sink; Event Hubs Capture archives raw Avro to ADLS for replay; Synapse or Power BI reads aggregates.

```python
events = spark.readStream.format("eventhubs").option("eventhubs.connectionString", connection_string).load()
(events.writeStream.format("delta")
 .option("checkpointLocation", "abfss://lake@acmedata.dfs.core.windows.net/checkpoints/telemetry")
 .trigger(availableNow=True)
 .toTable("curated.telemetry"))
```

Why: Event Hubs absorbs bursty device traffic with ordering per partition key, Structured Streaming gives exactly-once Delta writes with checkpointing, and Capture keeps the raw stream available for reprocessing after a model change.

### Use case 3: Governed self-service analytics with Purview and Unity Catalog

Business context: a bank wants analysts to discover data without exposing PII, under regulatory audit. Data volume: ~50 TB across 300 tables.

Architecture: ADLS Gen2 registered in Unity Catalog, Purview scanning and classifying PII, Entra ID groups mapped to Unity Catalog grants, and Power BI consuming curated marts.

```bash
az role assignment create --assignee <analyst-group-id> --assignee-principal-type Group \
  --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata
```

Why: Unity Catalog centralizes table and column grants and lineage, Purview provides classification and data mapping for compliance, and Entra ID groups keep access management aligned with HR-driven identity.

### Use case 4: Cost-controlled log analytics with serverless SQL

Business context: an SRE team needs 12 months of application logs for incident response without a standing warehouse. Data volume: ~30 TB, bursty query patterns.

Architecture: diagnostic settings land logs in ADLS Gen2 as Parquet, Synapse SQL serverless queries them in place, and Workload Groups plus a per-query cost ceiling cap spend.

```sql
SELECT level, count(*) AS events
FROM OPENROWSET(BULK 'https://acmedata.dfs.core.windows.net/lake/logs/app/*/*.parquet', FORMAT = 'PARQUET') AS logs
WHERE status >= 500
GROUP BY level;
```

Why: serverless SQL avoids a provisioned warehouse for intermittent queries, Parquet plus directory partitioning keeps scans small, and lifecycle policies move older logs to cooler tiers.

## Azure exercises

These use only the standard library to model sizing and layout decisions. Prices are illustrative; treat each assert as a check on the formula.

### Try it 4. Choose ADLS access tiers over 12 months

**Task.** Assign each dataset to Hot, Cool, Cold, or Archive from its days since last read, then compare 12 months of storage cost with keeping everything Hot. Use illustrative prices per GB per month: Hot 0.02, Cool 0.01, Cold 0.005, Archive 0.002.

**Hint.** Archive needs at least 180 days since last read, Cold at least 90, and Cool at least 30. Anything read more recently stays Hot. Remember that moving a blob out of a tier before its minimum days incurs an early deletion charge.

**Expected result.** `raw_events` archives, `curated_sales` goes Cold, `gold_marts` goes Cool, and `live_landing` stays Hot. The tiered monthly cost is about 690 against 4,940 for all-Hot, a saving of about 4,250 per month.

In [ ]:
PRICE_PER_GB_MONTH = {"hot": 0.02, "cool": 0.01, "cold": 0.005, "archive": 0.002}
MIN_DAYS = {"hot": 0, "cool": 30, "cold": 90, "archive": 180}


def choose_tier(days_since_read):
    for tier in ("archive", "cold", "cool", "hot"):
        if days_since_read >= MIN_DAYS[tier]:
            return tier
    return "hot"


datasets = [
    ("raw_events", 200_000, 210),
    ("curated_sales", 40_000, 95),
    ("gold_marts", 5_000, 45),
    ("live_landing", 2_000, 0),
]

tiered = 0.0
all_hot = 0.0
for name, size_gb, days in datasets:
    tier = choose_tier(days)
    tiered += size_gb * PRICE_PER_GB_MONTH[tier]
    all_hot += size_gb * PRICE_PER_GB_MONTH["hot"]
    print(f"{name:<14} last read {days:>3}d -> {tier}")

saving = all_hot - tiered
print(f"tiered = {tiered:,.0f} vs all-hot = {all_hot:,.0f} per month")
print(f"saving = {saving:,.0f} per month")

assert choose_tier(210) == "archive"
assert choose_tier(95) == "cold"
assert choose_tier(45) == "cool"
assert choose_tier(0) == "hot"
assert round(tiered) == 690
assert round(saving) == 4250

### Try it 5. Detect skew across 60 distributions

**Task.** A fact table has one row count per distribution. Compute the mean, the ratio of the largest distribution to the mean, and flag skew when the ratio exceeds 1.5.

**Hint.** Build the list with 59 even distributions and one hot distribution, then use `max`, `sum`, and `len`.

**Expected result.** The mean is about 1,016,667 rows, the hot distribution has 2,000,000 rows, and the ratio is about 1.967, so the table is skewed.

In [ ]:
rows_per_distribution = [1_000_000] * 59 + [2_000_000]
DISTRIBUTION_COUNT = 60
SKEW_THRESHOLD = 1.5

total_rows = sum(rows_per_distribution)
mean_rows = total_rows / len(rows_per_distribution)
max_rows = max(rows_per_distribution)
ratio = max_rows / mean_rows
skewed = ratio > SKEW_THRESHOLD
hottest = rows_per_distribution.index(max_rows)

print(f"distributions = {len(rows_per_distribution)}")
print(f"mean = {mean_rows:,.0f} rows, max = {max_rows:,} rows on distribution {hottest}")
print(f"max/mean = {ratio:.3f}, skewed = {skewed}")

assert len(rows_per_distribution) == DISTRIBUTION_COUNT
assert round(ratio, 3) == 1.967
assert skewed is True
assert hottest == 59

### Try it 6. Size a Synapse DWU from concurrency

**Task.** Given the maximum concurrent queries documented per performance level, pick the smallest level that supports a peak of 30 concurrent queries, and the smallest that supports 90.

**Hint.** Sort the levels by concurrency and take the first one whose maximum is at least the peak. Remember that scaling up raises DWUs and lowers the per-query share for small levels, so concurrency does not grow linearly forever.

**Expected result.** 30 concurrent queries need DW1000c (32 maximum). 90 concurrent queries need DW6000c (128 maximum), because DW3000c tops out at 64.

In [ ]:
MAX_CONCURRENT_QUERIES = {
    "DW100c": 4,
    "DW200c": 8,
    "DW300c": 12,
    "DW400c": 16,
    "DW500c": 20,
    "DW1000c": 32,
    "DW2000c": 48,
    "DW3000c": 64,
    "DW6000c": 128,
}


def smallest_level(peak_concurrency):
    for level, limit in sorted(MAX_CONCURRENT_QUERIES.items(), key=lambda item: (item[1], item[0])):
        if limit >= peak_concurrency:
            return level, limit
    raise ValueError("beyond the largest level")


level_30, limit_30 = smallest_level(30)
level_90, limit_90 = smallest_level(90)
print(f"peak 30 -> {level_30} (max {limit_30})")
print(f"peak 90 -> {level_90} (max {limit_90})")

assert level_30 == "DW1000c"
assert level_90 == "DW6000c"
assert MAX_CONCURRENT_QUERIES["DW3000c"] < 90

### Try it 8. Cost a dedicated pool with pause

**Task.** A dedicated SQL pool costs an illustrative 6.00 per hour at DW1000c. Compare always-on (720 hours a month) with pausing outside a 10-hour business window on 22 working days. Report the monthly cost and the saving.

**Hint.** Running hours are 10 times 22. The pool bills by the hour while it is online, and there is no charge while it is paused, though a paused pool must be resumed before the next query.

**Expected result.** Always-on costs 4,320; business-hours-only costs 1,320, a saving of 3,000, or about 69 percent.

In [ ]:
RATE_PER_HOUR = 6.00
HOURS_PER_MONTH = 720
BUSINESS_HOURS = 10 * 22

always_on = HOURS_PER_MONTH * RATE_PER_HOUR
paused = BUSINESS_HOURS * RATE_PER_HOUR
saving = always_on - paused
saving_pct = saving / always_on * 100

print(f"always-on   = {always_on:,.0f} per month")
print(f"paused      = {paused:,.0f} per month ({BUSINESS_HOURS} hours)")
print(f"saving      = {saving:,.0f} per month ({saving_pct:.1f}%)")

assert always_on == 4320
assert paused == 1320
assert saving == 3000
assert round(saving_pct, 1) == 69.4